# Tidal Aliasing Analysis – Workflow Notebook

End-to-end notebook for the project *Quantifying Satellite Tidal Aliasing Bias for Intertidal Mapping Along the Macrotidal Korean Coast*.

Stages:
1. Load site configuration
2. Pull Landsat/Sentinel-2 metadata from Earth Engine
3. Compute FES2014 tide heights at acquisition times and a dense reference series
4. (Optional) Cross-validate FES2014 against nearest KHOA tide-gauge observations
5. Compute aliasing statistics (spread, offsets, KS, chi-squared)
6. Generate publication figures

**Note (published-results pipeline)**: The figures and statistics reported in the manuscript use the KHOA reference path (Section 2.2). The FES2014 `synthetic_reference_series` workflow shown in stage 2 of this notebook is retained as an alternative pipeline for astronomical-only sensitivity analysis (Section 5.4 (ii) and Section 4.7) and was not used to produce the headline numbers.

Run each section interactively, or use the headline script `scripts/run_aliasing_pipeline.py` for non-interactive batch runs.

In [ ]:
import sys, os
from pathlib import Path

PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import load_sites, load_settings, resolve_path
sites = load_sites()
settings = load_settings()
for s in sites:
    print(f"{s.id:10s} | {s.name_en:18s} | lon={s.lon:.3f} lat={s.lat:.3f} | tidal range ~{s.tidal_range_m} m")

## 1. GEE metadata extraction

Requires `EE_PROJECT` env var set to your Earth Engine Cloud project id and `earthengine authenticate` already run.

In [ ]:
from src.gee.auth import initialize
from src.gee.metadata import extract_site_metadata, save_metadata

initialize()  # uses EE_PROJECT env var

gee_dir = resolve_path(settings['paths']['gee_metadata'])
start = settings['time_period']['start']
end = settings['time_period']['end']
sensors = ['L5', 'L7', 'L8', 'L9', 'S2']

for site in sites:
    out = gee_dir / f'{site.id}_scenes.parquet'
    if out.exists():
        print(f'cached: {out}')
        continue
    df = extract_site_metadata(site, sensors, start, end)
    if df.empty:
        print(f'  no scenes for {site.id}')
        continue
    save_metadata(df, gee_dir, site.id)
    print(f'{site.id}: {len(df)} scenes')

### Quick sanity check

In [ ]:
site = sites[0]
df = pd.read_parquet(gee_dir / f'{site.id}_scenes.parquet')
df['year'] = pd.to_datetime(df['datetime_utc']).dt.year
print(df.groupby(['sensor']).size())
df.pivot_table(index='year', columns='sensor', aggfunc='size', fill_value=0).plot.bar(stacked=True, figsize=(10,4))
plt.title(f'Scene count by year – {site.name_en}'); plt.tight_layout()

## 2. FES2014 tide computation

**Prerequisite**: download FES2014 ocean-tide NetCDF files from AVISO and place them under `data/raw/fes2014/ocean_tide/` (with optional `ocean_tide_extrapolated/` for coastal cells).

In [ ]:
from src.tides.fes2014 import compute_tide_heights, synthetic_reference_series

fes_dir = resolve_path(settings['paths']['fes2014'])
processed = resolve_path(settings['paths']['processed'])

ref_cfg = settings['aliasing']['reference_synthetic']

for site in sites:
    scenes_path = gee_dir / f'{site.id}_scenes.parquet'
    out_scenes = processed / f'{site.id}_scenes_with_tide.parquet'
    out_ref = processed / f'{site.id}_reference.parquet'
    if not scenes_path.exists():
        continue

    scenes = pd.read_parquet(scenes_path)
    if not out_scenes.exists():
        scenes['tide_m'] = compute_tide_heights(site.lon, site.lat, scenes['datetime_utc'], fes_dir)
        scenes.to_parquet(out_scenes, index=False)
    if not out_ref.exists():
        t0 = pd.to_datetime(scenes['datetime_utc']).min()
        t1 = t0 + pd.Timedelta(days=int(365.25 * ref_cfg['years']))
        ref = synthetic_reference_series(site.lon, site.lat, t0, t1, ref_cfg['sampling_minutes'], fes_dir)
        ref.to_parquet(out_ref, index=False)
    print(f"{site.id}: scenes={len(scenes)} ref ok")

## 3. (Optional) Cross-validate FES2014 against KHOA observations

Requires `KHOA_API_KEY` env var. Cached daily fetches are stored under `data/raw/khoa/`.

In [ ]:
from datetime import date
from src.tides.khoa import fetch_tide_hourly_range, interpolate_at_times, diagnose

# Step A: One-time diagnostic to verify API endpoint URL and key.
info = diagnose(api_id='tide_hourly', station_code='DT_0001', day=date(2024, 6, 1))
print(f"HTTP {info['status_code']}, rows: {info['rows_parsed']}")
print(f"URL : {info['request_url']}")
if info['sample_row']:
    print(f"Sample row: {info['sample_row']}")
else:
    print(f"Raw head : {info['raw_head'][:300]}")
    print('\n[!] rows_parsed == 0 → check config/khoa_apis.yaml URL or ServiceKey.')

In [ ]:
khoa_dir = resolve_path(settings['paths']['khoa'])
site = sites[0]
station = site.khoa_stations[0]

obs = fetch_tide_hourly_range(station.code, date(2024, 1, 1), date(2024, 12, 31), khoa_dir)
print(f'{station.name_en} ({station.code}): {len(obs)} hourly observations')

scenes = pd.read_parquet(processed / f'{site.id}_scenes_with_tide.parquet')
sub = scenes[(scenes['datetime_utc'] >= '2024-01-01') & (scenes['datetime_utc'] <= '2024-12-31')].copy()
sub['khoa_m'] = interpolate_at_times(obs, sub['datetime_utc']).values
sub = sub.dropna(subset=['khoa_m'])

diff = sub['tide_m'] - sub['khoa_m']
print(f'FES2014 − KHOA: bias={diff.mean():+.3f} m, RMSE={(diff**2).mean()**0.5:.3f} m, n={len(diff)}')

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(sub['khoa_m'], sub['tide_m'], s=10, alpha=0.6)
lim = (min(sub.khoa_m.min(), sub.tide_m.min()) - 0.5, max(sub.khoa_m.max(), sub.tide_m.max()) + 0.5)
ax.plot(lim, lim, 'k--', linewidth=0.8)
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel('KHOA observed tide (m)')
ax.set_ylabel('FES2014 predicted tide (m)')
ax.set_title(f'{station.name_en} 2024 – FES2014 vs KHOA')
ax.grid(alpha=0.3)
plt.tight_layout()

## 4. Tidal aliasing statistics

In [ ]:
from src.analysis.aliasing import stats_table

all_scenes = []
references = {}
for site in sites:
    sp = processed / f'{site.id}_scenes_with_tide.parquet'
    rp = processed / f'{site.id}_reference.parquet'
    if not (sp.exists() and rp.exists()):
        continue
    all_scenes.append(pd.read_parquet(sp))
    references[site.id] = pd.read_parquet(rp)['tide_m'].to_numpy()

combined = pd.concat(all_scenes, ignore_index=True)
stats = stats_table(combined, references, groupby=['site_id', 'sensor'], n_bins=settings['aliasing']['histogram_bins'])
tables_dir = resolve_path(settings['paths']['tables'])
tables_dir.mkdir(parents=True, exist_ok=True)
stats.to_csv(tables_dir / 'aliasing_stats.csv', index=False)
stats

## 5. Publication figures

In [ ]:
from src.visualization.plots import plot_spread_offset, plot_temporal_evolution, plot_tide_distribution

figs_dir = resolve_path(settings['paths']['figures'])

for site in sites:
    sp = processed / f'{site.id}_scenes_with_tide.parquet'
    rp = processed / f'{site.id}_reference.parquet'
    if not (sp.exists() and rp.exists()):
        continue
    scenes = pd.read_parquet(sp)
    ref = pd.read_parquet(rp)['tide_m'].to_numpy()
    plot_tide_distribution(scenes, ref, site.name_en, out_path=figs_dir / f'distribution_{site.id}.png')
    plot_temporal_evolution(scenes, site.name_en, out_path=figs_dir / f'temporal_{site.id}.png')

plot_spread_offset(stats, out_path=figs_dir / 'spread_offsets.png')
plt.show()